# 49 — Online direct-U20-gradient pilot analysis

This notebook combines workers 48-0 through 48-3. It exact-matches the unrefined baseline, true U20-gradient update, and equal-RMS/equal-compute random control by suite, task, initialization index, and initialization hash.

`REQUIRE_FULL_COHORT=False` is safe for an interim preview: only identities with all three completed arms enter any SR denominator, and incomplete coverage is printed. Set it to `True` for the final 220-identity analysis.

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## Configuration and imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analysis.uncertainty_gradient import (
    arm_success_table, fetch_direct_gradient_rows, first_u20_gate_sweep,
    gradient_telemetry_table, match_direct_gradient_cohort,
    paired_effect_table, suite_effect_table, telemetry_by_outcome_transition)
from pnp.uncertainty_gradient_experiment import (
    DIRECT_U20_GRADIENT_EXPERIMENT, DIRECT_U20_GRADIENT_RMS)
from pnp.store import SupabaseStore

EXPECTED_IDENTITIES = 220
REQUIRE_FULL_COHORT = False  # switch to True after all four workers finish
OUTPUT = Path('direct_u20_gradient_pro220_outputs')
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

## Load exact configurations and form matched three-arm identities

In [ ]:
rows = fetch_direct_gradient_rows(
    store, experiment=DIRECT_U20_GRADIENT_EXPERIMENT,
    step_size=DIRECT_U20_GRADIENT_RMS)
paired, coverage = match_direct_gradient_cohort(
    rows, expected_identities=EXPECTED_IDENTITIES,
    require_complete=REQUIRE_FULL_COHORT)

mode = 'FINAL STRICT ANALYSIS' if REQUIRE_FULL_COHORT else 'INTERIM MATCHED PREVIEW'
print(mode)
print(f'Using {len(paired)} identities with all three arms complete; '
      f'{len(paired) * 3} matched rollouts.')
display(coverage)
if not REQUIRE_FULL_COHORT:
    print('Preview caveat: rerun with REQUIRE_FULL_COHORT=True when all workers finish.')
paired.to_csv(OUTPUT / 'matched_episode_table.csv', index=False)
coverage.to_csv(OUTPUT / 'coverage.csv', index=False)

## Primary result: paired success-rate effects

Every delta below uses all currently matched identities. `F_to_S` and `S_to_F` are paired outcome flips, not independent rollout counts. The random-control comparison determines whether the gradient direction contributes beyond an arbitrary latent movement of the same RMS and compute.

In [ ]:
success = arm_success_table(paired)
effects = paired_effect_table(paired)
print('Success rates on the exact matched cohort')
display(success)
print('Paired comparisons with identity-bootstrap 95% intervals')
display(effects[[
    'comparison', 'matched_episodes', 'baseline_sr_pct', 'condition_sr_pct',
    'condition_minus_baseline_pp', 'delta_ci_low_pp', 'delta_ci_high_pp',
    'F_to_S', 'S_to_F', 'paired_p_value']])
success.to_csv(OUTPUT / 'arm_success_rates.csv', index=False)
effects.to_csv(OUTPUT / 'paired_effects.csv', index=False)

In [ ]:
suite_effects = suite_effect_table(paired)
suite_sr = (paired.groupby('suite', sort=True)
    .agg(episodes=('suite', 'size'),
         baseline_sr=('baseline_success', 'mean'),
         gradient_sr=('gradient_success', 'mean'),
         random_sr=('random_success', 'mean')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
x = np.arange(len(suite_sr)); width = .26
labels = suite_sr.suite.str.removeprefix('libero_')
axes[0].bar(x - width, 100 * suite_sr.baseline_sr, width,
            label='unrefined baseline', color='#4C78A8')
axes[0].bar(x, 100 * suite_sr.gradient_sr, width,
            label='true U20 gradient', color='#F58518')
axes[0].bar(x + width, 100 * suite_sr.random_sr, width,
            label='random control', color='#9D755D')
axes[0].set_xticks(x, labels, rotation=40, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0, 105),
            title='Matched success rate by suite')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)

for offset, comparison, color, label in (
        (-.18, 'gradient minus baseline', '#F58518', 'gradient − baseline'),
        (.18, 'random minus baseline', '#9D755D', 'random − baseline')):
    group = (suite_effects[suite_effects.comparison.eq(comparison)]
             .set_index('suite').reindex(suite_sr.suite))
    axes[1].bar(x + offset, group.condition_minus_baseline_pp, .36,
                color=color, label=label)
axes[1].axhline(0, color='black', linewidth=1)
gradient_overall = effects.loc[
    effects.comparison.eq('gradient minus unrefined baseline'),
    'condition_minus_baseline_pp'].iloc[0]
axes[1].axhline(gradient_overall, color='#9467BD', linestyle='--',
                label=f'overall gradient {gradient_overall:+.2f} pp')
axes[1].set_xticks(x, labels, rotation=40, ha='right')
axes[1].set(ylabel='Paired SR change (percentage points)',
            title='Whole-matched-cohort change by suite')
axes[1].legend(); axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'success_and_suite_deltas.png', dpi=180)
plt.show()
suite_effects.to_csv(OUTPUT / 'suite_effects.csv', index=False)

## Did the online gradient perform its local objective?

A useful gradient arm should lower common-noise U20 substantially more often than the random control. This is separate from whether lower local U20 improves task success.

In [ ]:
telemetry = gradient_telemetry_table(paired)
print('Online intervention telemetry')
display(telemetry)
print('First pre-update U20 should match between gradient and random arms before either intervenes:')
display(pd.DataFrame([{
    'matched_episodes': len(paired),
    'max_abs_difference': paired.first_pre_u20_abs_difference.max(),
    'mean_abs_difference': paired.first_pre_u20_abs_difference.mean(),
}]))
telemetry.to_csv(OUTPUT / 'gradient_telemetry_summary.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
colors = ['#F58518', '#9D755D']
axes[0].bar(telemetry.arm, telemetry.mean_post_minus_pre_u20, color=colors)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set(ylabel='Mean post-U20 minus pre-U20', title='Local objective change')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(axis='y', alpha=.2)

axes[1].hist(paired.gradient_mean_delta_u20.dropna(), bins=25, alpha=.65,
             color='#F58518', label='true gradient')
axes[1].hist(paired.random_mean_delta_u20.dropna(), bins=25, alpha=.55,
             color='#9D755D', label='random control')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set(xlabel='Episode-mean local U20 change', ylabel='Episodes',
            title='Distribution across episodes')
axes[1].legend(); axes[1].grid(alpha=.2)

changed = paired.gradient_success.astype(int) - paired.baseline_success.astype(int)
palette = {-1: '#E45756', 0: '#A0A0A0', 1: '#54A24B'}
for value, label in ((-1, 'S→F'), (0, 'same outcome'), (1, 'F→S')):
    mask = changed.eq(value)
    axes[2].scatter(paired.loc[mask, 'gradient_first_pre_u20'],
                    paired.loc[mask, 'gradient_mean_delta_u20'],
                    alpha=.7, s=35, color=palette[value], label=label)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set(xlabel='Initial pre-update U20', ylabel='Episode-mean local U20 change',
            title='Local U reduction versus outcome flip')
axes[2].legend(); axes[2].grid(alpha=.2)
fig.savefig(OUTPUT / 'gradient_local_objective.png', dpi=180)
plt.show()

## Do larger uncertainty reductions correspond to useful outcome changes?

In [ ]:
transitions = telemetry_by_outcome_transition(paired)
display(transitions)
transitions.to_csv(OUTPUT / 'telemetry_by_outcome_transition.csv', index=False)

quantile_frame = paired[np.isfinite(paired.gradient_first_pre_u20)].copy()
bins = min(5, quantile_frame.gradient_first_pre_u20.nunique())
quantile_frame['initial_u_bin'] = pd.qcut(
    quantile_frame.gradient_first_pre_u20, q=bins, duplicates='drop')
u_strata = (quantile_frame.groupby('initial_u_bin', observed=True, sort=True)
    .agg(episodes=('suite', 'size'),
         mean_initial_u20=('gradient_first_pre_u20', 'mean'),
         baseline_sr=('baseline_success', 'mean'),
         gradient_sr=('gradient_success', 'mean'),
         mean_local_u20_change=('gradient_mean_delta_u20', 'mean')).reset_index())
u_strata['gradient_minus_baseline_pp'] = 100 * (
    u_strata.gradient_sr - u_strata.baseline_sr)
display(u_strata)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
transition_order = ['F→F', 'F→S', 'S→F', 'S→S']
plot_transition = transitions.set_index('transition').reindex(transition_order).dropna(how='all')
axes[0].bar(plot_transition.index, plot_transition.mean_local_u20_change,
            color=['#A0A0A0', '#54A24B', '#E45756', '#4C78A8'][:len(plot_transition)])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set(ylabel='Mean local U20 change', title='Gradient effect by outcome transition')
axes[0].grid(axis='y', alpha=.2)
axes[1].bar(np.arange(len(u_strata)), u_strata.gradient_minus_baseline_pp,
            color=np.where(u_strata.gradient_minus_baseline_pp >= 0, '#54A24B', '#E45756'))
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(np.arange(len(u_strata)),
                   [f'{value:.3f}' for value in u_strata.mean_initial_u20])
axes[1].set(xlabel='Mean initial U20 within quintile',
            ylabel='Gradient minus baseline SR (pp)',
            title='Is the gradient useful only at high initial U?')
axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'outcome_transition_and_u_strata.png', dpi=180)
plt.show()
u_strata.to_csv(OUTPUT / 'initial_u20_strata.csv', index=False)

## Exploratory high-U gate

This post-hoc rule uses the true-gradient rollout only when the first pre-intervention U20 exceeds a threshold; otherwise it keeps the matched baseline outcome. Every row retains the entire matched cohort in the SR denominator. Because thresholds are selected and evaluated on the same 220 identities, treat the best result as a hypothesis for a new held-out run—not a final performance estimate.

In [ ]:
gate = first_u20_gate_sweep(paired, grid_size=41)
top_gate = gate.sort_values(
    ['gated_minus_baseline_pp', 'episodes_using_gradient'],
    ascending=[False, True]).head(10)
display(top_gate[[
    'threshold', 'episodes_in_sr_denominator', 'episodes_using_gradient',
    'baseline_sr_pct', 'gated_policy_sr_pct', 'gated_minus_baseline_pp',
    'delta_ci_low_pp', 'delta_ci_high_pp', 'selected_F_to_S', 'selected_S_to_F']])
gate.to_csv(OUTPUT / 'initial_u20_gate_sweep.csv', index=False)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(gate.threshold, gate.gated_minus_baseline_pp,
        marker='o', color='#F58518', label='gated gradient SR change')
ax.fill_between(gate.threshold, gate.delta_ci_low_pp, gate.delta_ci_high_pp,
                color='#F58518', alpha=.15, label='paired 95% CI')
ax.axhline(0, color='black', linewidth=1)
ax.set(xlabel='Initial U20 threshold: apply gradient when U20 ≥ threshold',
       ylabel='Whole-matched-cohort SR change (pp)',
       title='Exploratory initial-U20 gate sweep')
ax.grid(alpha=.2)
other = ax.twinx()
other.plot(gate.threshold, gate.episodes_using_gradient,
           color='#4C78A8', linestyle='--', label='episodes using gradient')
other.set_ylabel('Episodes using gradient')
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = other.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2, loc='best')
fig.tight_layout(); fig.savefig(OUTPUT / 'initial_u20_gate_sweep.png', dpi=180)
plt.show()

## Compact decision summary

In [ ]:
gradient_effect = effects[effects.comparison.eq(
    'gradient minus unrefined baseline')].iloc[0]
random_effect = effects[effects.comparison.eq(
    'random minus unrefined baseline')].iloc[0]
direction_effect = effects[effects.comparison.eq(
    'gradient minus random control')].iloc[0]
gradient_local = telemetry[telemetry.arm.eq('true U20 gradient')].iloc[0]
random_local = telemetry[telemetry.arm.eq('random control')].iloc[0]

print(f"Matched identities: {len(paired)}/{EXPECTED_IDENTITIES}")
print(f"Gradient vs baseline: {gradient_effect.condition_minus_baseline_pp:+.2f} pp "
      f"(F→S {gradient_effect.F_to_S}, S→F {gradient_effect.S_to_F})")
print(f"Random vs baseline:   {random_effect.condition_minus_baseline_pp:+.2f} pp")
print(f"Gradient vs random:   {direction_effect.condition_minus_baseline_pp:+.2f} pp")
print(f"Mean local U20 change: gradient {gradient_local.mean_post_minus_pre_u20:+.6f}; "
      f"random {random_local.mean_post_minus_pre_u20:+.6f}")
print('\nInterpretation order:')
print('1. Confirm the gradient lowers local U20 more than random.')
print('2. Check whether gradient beats both baseline and random, with paired uncertainty.')
print('3. If overall neutral, inspect high-U strata/gating before changing step size.')
print('4. Treat the best same-data gate only as a follow-up hypothesis.')
print('\nOutputs:', OUTPUT.resolve())